# Goal: statistical-summary 

In [1]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
Date: 10-10-2026 14:42 IST
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\nDate: 10-10-2026 14:42 IST\n'

In [2]:
import polars as pl

In [3]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-08\data\lev-08_merged.parquet"
pdf = pl.scan_parquet(path)

In [4]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('Item_Code_8_1', String),
        ('Out_of_home_qty_000', Float64),
        ('Out_of_home_value_rs', Float64),
        ('Total_consumption_qty_000', Float64),
        ('Total_consumption_value_rs', Int64),
        ('Source_8_1', String),
        ('Multiplier', Int64)])

# Useful Variables

In [5]:
cols = [
'Out_of_home_qty_000',
'Out_of_home_value_rs',
'Total_consumption_qty_000',
'Total_consumption_value_rs',
'Source_8_1',
'Multiplier',
]


In [6]:
df = pdf.select(cols)

In [7]:
df.head(2).collect()

Out_of_home_qty_000,Out_of_home_value_rs,Total_consumption_qty_000,Total_consumption_value_rs,Source_8_1,Multiplier
f64,f64,f64,i64,str,i64
null,null,50.0,200,"""1""",57986
null,null,70.0,329,"""""",57986


In [8]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in cols]
)

In [9]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

Out_of_home_qty_000,Out_of_home_value_rs,Total_consumption_qty_000,Total_consumption_value_rs,Source_8_1,Multiplier
u32,u32,u32,u32,u32,u32
362,1065,913,5344,8,23565


# Logic

In [10]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_22976\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [11]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
Out_of_home_qty_000,566964.0,2432978.0,12.328296,42.9515,0.0,0.0,0.0,0.0,900.0
Out_of_home_value_rs,703546.0,2296396.0,110.296364,234.198249,0.0,0.0,0.0,120.0,6000.0
Total_consumption_qty_000,2476382.0,523560.0,37.783739,65.35093,0.0,4.0,10.0,50.0,9615.0
Total_consumption_value_rs,2999942.0,0.0,458.930211,603.828908,1.0,16.0,235.0,708.0,16393.0
Multiplier,2999942.0,0.0,111815.582719,75076.534198,369.0,59424.0,115804.0,150927.0,2366902.0


# Categorical Columns

In [12]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))

Source_8_1


Source_8_1,count
i32,u32
null,1536854
1,1228602
4,121624
2,94914
3,10524
9,5342
6,1326
5,756
